<a href="https://colab.research.google.com/github/charlesbihdev/charlesbihdev-ML-DataScience-Notebooks/blob/final-year-face-detection-model-train/final_year_face_detection_model_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install facenet-pytorch pillow


In [2]:
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
import torch
import os
import numpy as np
import sqlite3


In [3]:
# Your image folder path
image_folder = '/content/drive/MyDrive/ml/my-images/train'

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load MTCNN and FaceNet
mtcnn = MTCNN(image_size=160, margin=0).to(device)
model = InceptionResnetV1(pretrained='vggface2').eval().to(device)


  0%|          | 0.00/107M [00:00<?, ?B/s]

In [4]:
conn = sqlite3.connect('/content/drive/MyDrive/ml/my-images/face_data.db')
conn.execute('''
  CREATE TABLE IF NOT EXISTS faces (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    embedding BLOB NOT NULL
  )
''')
conn.commit()
conn.close()


In [5]:
def save_embedding_to_db(name, embedding):
    conn = sqlite3.connect('/content/drive/MyDrive/ml/my-images/face_data.db')
    cursor = conn.cursor()
    blob = embedding.astype(np.float32).tobytes()
    cursor.execute("INSERT INTO faces (name, embedding) VALUES (?, ?)", (name, blob))
    conn.commit()
    conn.close()


In [6]:
person_name = "Charles"
image_names = ["img1.jpg", "img2.jpg", "img3.jpg", "img4.JPG", "img5.PNG"]
embeddings = []

for img_name in image_names:
    img_path = os.path.join(image_folder, img_name)
    img = Image.open(img_path).convert('RGB')

    face = mtcnn(img)
    if face is not None:
        with torch.no_grad():
            emb = model(face.unsqueeze(0).to(device)).cpu().numpy()
            embeddings.append(emb[0])
    else:
        print(f"Face not detected in {img_name}")


In [7]:
if embeddings:
    average_embedding = np.mean(embeddings, axis=0)
    save_embedding_to_db(person_name, average_embedding)
    print("✅Face embedding saved for:", person_name)
else:
    print("❌No embeddings generated.")


✅Face embedding saved for: Charles
